# Module 098: Performance Optimization & Profiling

Phase 10: Concurrency & Internals | Duration: 2 hours

## Profiling with cProfile

In [ ]:
import cProfile
import pstats
from typing import List

def compute(n: int) -> int:
    total: int = 0
    for i in range(n):
        total += i ** 2 + i * 2
    return total

def run_computations() -> None:
    results: List[int] = []
    for n in [10000, 50000, 100000]:
        results.append(compute(n))
    print(f"Results: {results}")

cProfile.run('run_computations()', 'profile_stats')

p = pstats.Stats('profile_stats')
p.sort_stats('cumtime').print_stats(10)

## Micro-benchmarks with timeit

In [ ]:
import timeit
from typing import Callable, Any

def benchmark(statement: str, setup: str = '', number: int = 100000) -> float:
    total: float = timeit.timeit(statement, setup=setup, number=number)
    return total / number

# Compare list vs set membership
setup_list: str = "data = list(range(1000))"
setup_set: str = "data = set(range(1000))"

list_time: float = benchmark("500 in data", setup_list)
set_time: float = benchmark("500 in data", setup_set)

print(f"List membership: {list_time:.8f}s")
print(f"Set membership:  {set_time:.8f}s")
print(f"Set is {list_time / set_time:.1f}x faster")

## Algorithmic Complexity (Big O)

| Operation | List | Set | Dict |
|-----------|------|-----|------|
| Lookup | O(n) | O(1) | O(1) |
| Insert | O(1)* | O(1) | O(1) |
| Delete | O(n) | O(1) | O(1) |
| Membership | O(n) | O(1) | O(1) |

*Amortized at end; O(n) at beginning/middle

## List vs Array vs Set Performance

In [ ]:
import timeit
import sys
import array
from typing import List

SIZE: int = 100000

# Memory comparison
py_list: List[int] = list(range(SIZE))
arr: array.array = array.array('i', range(SIZE))

print(f"List size:   {sys.getsizeof(py_list) + SIZE * 28} bytes (approx)")
print(f"Array size:  {sys.getsizeof(arr) + SIZE * 4} bytes (approx)")

# Operation time comparison
setup_list: str = "l = list(range(10000))"
setup_arr: str = "import array; a = array.array('i', range(10000))"
setup_set: str = "s = set(range(10000))"

for name, stmt, setup in [
    ("List iteration", "for x in l: pass", setup_list),
    ("Array iteration", "for x in a: pass", setup_arr),
    ("Set comprehension", "{x for x in s}", setup_set),
]:
    t: float = timeit.timeit(stmt, setup, number=1000)
    print(f"{name}: {t:.4f}s")

## Identifying Bottlenecks

1. **Profile first** - Don't guess, use cProfile
2. **Focus on hot spots** - Functions with highest cumtime
3. **Algorithm improvements** - Better data structures (set vs list)
4. **Use __slots__** - Reduce memory per instance
5. **Consider C extensions** - Cython for CPU-bound code
6. **Try PyPy** - JIT compiler can speed up Python code significantly

```
Rule of thumb:
  - Make it work first
  - Profile to find bottlenecks
  - Optimize the 20% that uses 80% of the time
  - Measure the improvement
```